# Assignment 3 — Hybrid Biomedical Image Analysis
## Fluorescence microscopy nuclei: VLM description, classical features, U-Net, hybrid pipeline

**Module:** 7PAM2032 Data Analysis with AI

This notebook builds the pipeline described in the assignment brief:

```
raw image -> segmentation -> quantitative region features
          -> structured JSON record -> short narrative
```

and runs it on the synthetic stained-nuclei dataset (256x256 DAPI-like
fluorescence microscopy, 80 train / 20 val / 12 test, with exact ground-truth
masks).

| Task | What it does | Section |
|---|---|---|
| 1 | Grayscale + resize + EDA, then a direct multimodal (VLM) description | 2-3 |
| 2 | Otsu + morphology + regionprops, then a numbers-first description | 4-5 |
| 3 | Train the small U-Net, evaluate with Dice and IoU | 6 |
| 4 | Full hybrid pipeline on the unseen test split -> CSV of JSON records | 7 |
| Ext | Robustness trace, loss ablation, vision-model comparison | 8-9 |

**Before you run anything:** set the runtime to a GPU
(*Runtime -> Change runtime type -> T4 GPU*). The U-Net trains on CPU in a few
minutes, but the vision model is painfully slow without a GPU.

> These outputs are for educational use only. None of the models here are
> validated for clinical use, and a vision-language model will produce
> confident, fluent descriptions of things that are not in the image.

## 1. Setup

Three cells: install Ollama and the Python client, start the server, pull the
models. Run them once per Colab session (Colab wipes everything on
disconnect). Total download is roughly 12 GB on a fresh session.

In [ ]:
# --- Setup 1 of 3: install the Ollama server and the Python client ---
# Two Colab-specific fixes:
#  * zstd: Colab's base image lacks it, and Ollama's installer needs it to unpack
#    its release archive ("This version requires zstd for extraction").
#  * lshw: without it the installer prints "Unable to detect NVIDIA/AMD GPU" and
#    skips the GPU libraries, so the models run on CPU and are far slower.
!apt-get update -qq && apt-get install -y -qq zstd lshw
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama scikit-image torch torchvision pandas matplotlib

# Verify. Expect a path and a version number. The "could not connect to a running
# Ollama instance" warning here is normal - the server is started in setup cell 2.
!which ollama && ollama --version

In [ ]:
# --- Setup 2 of 3: start the Ollama server in the background ---
# Colab has no systemd, so nothing auto-starts the server. Spawn it, then poll
# /api/tags once a second until it answers.
import os, subprocess, time, urllib.request

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

def ollama_up(timeout=1):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=timeout)
        return True
    except Exception:
        return False

if not ollama_up():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for attempt in range(60):
        if ollama_up():
            print(f"Ollama server ready (after {attempt + 1}s)")
            break
        time.sleep(1)
    else:
        print("Ollama did not start in 60s. Re-run this cell.")
else:
    print("Ollama already running.")

In [ ]:
# --- Setup 3 of 3: pull the models ---
# llama3.2-vision (~7.9 GB): primary multimodal model for Task 1.
# qwen2.5vl:7b   (~6.0 GB): backup vision model. Some Ollama builds reject
#                           llama3.2-vision with "unknown model architecture:
#                           mllama"; qwen2.5vl uses a different architecture and
#                           is unaffected. It also doubles as the second model in
#                           the vision-comparison extension.
# llama3.2       (~2.0 GB): text narrator for Tasks 2 and 4.
# moondream      (~1.7 GB): small vision model, third point of comparison.
!ollama pull llama3.2-vision
!ollama pull qwen2.5vl:7b
!ollama pull llama3.2
!ollama pull moondream
!ollama list

In [ ]:
# --- Imports, seeds, device ---
import json, re, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from ollama import chat
from skimage import filters as skfilters, measure as skmeasure, morphology as skmorphology
from skimage.color import rgb2gray
from skimage.morphology import disk
from skimage.transform import resize
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The assignment asks for a common 256x256 analysis size (the dataset is
# already native 256). The U-Net trains at 128, as in Lab 4: at 256 a single
# run costs ~10x more, which makes the loss ablation impractical. Predicted
# masks are resized back to 256 before regionprops so that object areas stay in
# the same pixel units as the ground truth.
ANALYSIS_SIZE = 256
UNET_SIZE = 128

for d in ["outputs/figures", "outputs/metrics", "outputs/records", "models"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
print("torch:", torch.__version__)

## 2. Task 1a — data preparation and EDA

Download the dataset, convert to grayscale, confirm the common 256x256 size,
and look at what we are actually dealing with.

**A note on the grayscale conversion.** These are DAPI-like images: the
generator puts the signal mostly in blue (B = I, G = 0.35I, R = 0.12I) and
`rgb2gray` weights blue at only 0.0721. The conversion is therefore a *linear
rescale* of the underlying intensity to about 0.35I — the three channels are
perfectly correlated by construction, so no information is lost and Otsu is
invariant to the rescale. What it does cost is dynamic range, which is why we
normalise per image before the U-Net sees anything.

In [ ]:
# --- Download and unpack the dataset ---
!wget -q https://github.com/Nickolay-K/Assingnment-3-dataset/raw/main/nuclei_dataset.zip
!unzip -qo nuclei_dataset.zip -d .
!ls nuclei_dataset

DATA = Path("nuclei_dataset")
meta = pd.read_csv(DATA / "metadata.csv")
print(meta.groupby(["split", "density"]).size().to_string())

In [ ]:
# --- Loading helpers ---
from imageio.v2 import imread

def to_unit_range(a):
    """Min-max normalise to [0, 1] float32; safe on a constant image."""
    a = np.asarray(a, dtype=np.float32)
    lo, hi = float(a.min()), float(a.max())
    return (a - lo) / (hi - lo) if hi - lo > 1e-8 else np.zeros_like(a)

def load_gray(path, size=ANALYSIS_SIZE, normalise=False):
    """PNG -> grayscale float32 in [0, 1] at the requested size."""
    arr = np.asarray(imread(path))
    if arr.ndim == 3:
        gray = rgb2gray(arr[..., :3])
    else:
        gray = arr.astype(np.float32)
        if gray.max() > 1.0:
            gray = gray / 255.0
    if gray.shape != (size, size):
        gray = resize(gray, (size, size), anti_aliasing=True, preserve_range=True)
    gray = np.asarray(gray, dtype=np.float32)
    return to_unit_range(gray) if normalise else np.clip(gray, 0, 1)

def load_mask(path, size=ANALYSIS_SIZE):
    """Binary mask PNG -> float32 {0., 1.} at the requested size."""
    m = np.asarray(imread(path)).astype(np.float32)
    if m.ndim == 3:
        m = m[..., 0]
    m = (m > 127).astype(np.float32)
    if m.shape != (size, size):
        m = resize(m, (size, size), order=0, anti_aliasing=False, preserve_range=True)
    return (m > 0.5).astype(np.float32)

def list_split(split):
    img_dir = DATA / split / "images"
    return [(p.stem, p, DATA / split / "masks" / p.name)
            for p in sorted(img_dir.glob("*.png"))]

print(len(list_split("train")), "train |", len(list_split("val")), "val |",
      len(list_split("test")), "test")

In [ ]:
# --- Figure 1: one example per density regime, image over ground truth ---
regimes = ["sparse", "normal", "dense", "clustered"]
train_meta = meta[meta.split == "train"]

fig, axes = plt.subplots(2, 4, figsize=(14, 7.2))
for j, reg in enumerate(regimes):
    row = train_meta[train_meta.density == reg].iloc[0]
    img = load_gray(DATA / "train" / "images" / f"{row.image_id}.png")
    msk = load_mask(DATA / "train" / "masks" / f"{row.image_id}.png")
    axes[0, j].imshow(img, cmap="gray", vmin=0, vmax=float(img.max()))
    axes[0, j].set_title(f"{reg}\n{row.image_id} - {int(row.n_objects)} nuclei", fontsize=10)
    axes[1, j].imshow(msk, cmap="gray")
    axes[1, j].set_title(f"ground truth ({row.area_fraction*100:.1f}% fg)", fontsize=9)
    axes[0, j].axis("off"); axes[1, j].axis("off")
fig.suptitle("Task 1 - grayscale nuclei at 256x256, one per density regime")
fig.tight_layout()
fig.savefig("outputs/figures/fig1_samples.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 2: intensity histograms and object counts ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

for reg, colour in zip(regimes, ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]):
    iid = train_meta[train_meta.density == reg].iloc[0].image_id
    img = load_gray(DATA / "train" / "images" / f"{iid}.png")
    axes[0].hist(img.ravel(), bins=80, range=(0, 0.5), histtype="step",
                 label=reg, color=colour, linewidth=1.4)
axes[0].set_yscale("log")
axes[0].set_xlabel("grayscale intensity"); axes[0].set_ylabel("pixel count (log)")
axes[0].set_title("(a) Whole-image histogram by regime"); axes[0].legend(fontsize=8)

fg, bg = [], []
for iid in train_meta.image_id.head(20):
    img = load_gray(DATA / "train" / "images" / f"{iid}.png")
    msk = load_mask(DATA / "train" / "masks" / f"{iid}.png").astype(bool)
    fg.append(img[msk]); bg.append(img[~msk])
fg, bg = np.concatenate(fg), np.concatenate(bg)
axes[1].hist(bg, bins=80, range=(0, 0.5), alpha=0.6, label="background",
             color="#8C8C8C", density=True)
axes[1].hist(fg, bins=80, range=(0, 0.5), alpha=0.6, label="nuclei",
             color="#4C72B0", density=True)
axes[1].set_xlabel("grayscale intensity"); axes[1].set_ylabel("density")
axes[1].set_title("(b) Foreground vs background"); axes[1].legend(fontsize=8)

axes[2].boxplot([meta[meta.density == r].n_objects.values for r in regimes],
                tick_labels=regimes)
axes[2].set_ylabel("nuclei per image (ground truth)")
axes[2].set_title("(c) Object count by regime")
fig.tight_layout()
fig.savefig("outputs/figures/fig2_histograms.png", dpi=160, bbox_inches="tight")
plt.show()

sep = (fg.mean() - bg.mean()) / np.sqrt(0.5 * (fg.var() + bg.var()))
print(f"Foreground/background separability (Cohen's d): {sep:.2f}")
print("A d this large means the intensity histogram is almost perfectly")
print("bimodal - which is exactly the regime Otsu was designed for. Remember")
print("this number when the U-Net fails to beat Otsu in Task 3.")

In [ ]:
# --- EDA summary table ---
rows = []
for split in ["train", "val", "test"]:
    stats = np.array([[load_gray(ip).mean(), load_gray(ip).std()]
                      for _, ip, _ in list_split(split)])
    sub = meta[meta.split == split]
    rows.append({"split": split, "n_images": len(sub), "size": "256x256",
                 "mean_intensity": round(float(stats[:, 0].mean()), 4),
                 "std_intensity": round(float(stats[:, 1].mean()), 4),
                 "gt_objects_mean": round(float(sub.n_objects.mean()), 1),
                 "gt_objects_min": int(sub.n_objects.min()),
                 "gt_objects_max": int(sub.n_objects.max()),
                 "gt_area_fraction": round(float(sub.area_fraction.mean()), 4)})
eda = pd.DataFrame(rows)
eda.to_csv("outputs/metrics/eda_summary.csv", index=False)
eda

## 3. Task 1b — the direct multimodal (VLM) description

We send a representative image to `llama3.2-vision` twice: once with a naive
prompt, once with a prompt engineered to anchor the model as *descriptive
rather than diagnostic*, force a JSON record, and explicitly permit
`"uncertain"`.

The design of the structured prompt, and why each piece is there:

| Element | Failure mode it addresses |
|---|---|
| Role + explicit "not a diagnosis" | Vision models volunteer diagnoses unprompted |
| Closed vocabulary per field | Free-text categories are not machine-comparable |
| `"uncertain"` declared a *correct* answer | Gives the model a legal way to decline instead of confabulating |
| "JSON only, no fences" | Any prose before the object breaks `json.loads` |
| "Do not count objects" | Counting from pixels is exactly what it cannot do reliably |

In [ ]:
# --- The two prompts (Task 1). These are quoted verbatim in the report. ---
PROMPT_T1_NAIVE = "What is this?"

PROMPT_T1_STRUCTURED = '''You are assisting a researcher by DESCRIBING a biomedical image.
You are not making a diagnosis and you must not suggest one.

Return ONLY valid JSON. No commentary, no markdown code fences, no text before
or after the object.

Fields:
- modality (string: "fluorescence microscopy", "brightfield microscopy",
  "histology", "fundus photograph", "x-ray", "ct", "mri", or "uncertain")
- tissue_type (string: short noun phrase, or "uncertain")
- notable_features (list of up to 3 short noun phrases describing only what is
  visible: e.g. "round bright objects", "dark background", "clustered objects")
- image_quality (string: "good", "acceptable", "poor", or "uncertain")

Rules:
- Describe only what is visible. Do NOT infer a disease, a stain name, a
  species, a magnification, or a scale bar unless it is literally visible.
- If you are not confident about a field, output "uncertain" (or an empty list
  for notable_features). "uncertain" is a correct answer, not a failure.
- Do not count objects; you are not being asked for a number.
- Output must parse with json.loads.'''

def parse_model_json(text):
    """Parse a JSON-only reply, stripping markdown fences if present (Lab 2)."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        if text.endswith("```"):
            text = text.rsplit("```", 1)[0]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                pass
        return {"error": "could not parse JSON"}

def ask_vision(prompt, image_path, model=None, temperature=0.0):
    r = chat(model=model or VISION_MODEL,
             messages=[{"role": "user", "content": prompt,
                        "images": [str(image_path)]}],
             options={"temperature": temperature})
    return r["message"]["content"]

DEMO_ID, DEMO_PATH, _ = list_split("train")[1]   # a 'normal' density image
print("demo image:", DEMO_ID)

# --- Pick a vision model that actually loads on this Ollama build ---
# Ollama versions before 0.4 raise "unknown model architecture: mllama" on
# llama3.2-vision. Rather than assume, send each candidate a one-token request
# with the real image and keep the first that answers.
VISION_CANDIDATES = ["llama3.2-vision", "qwen2.5vl:7b", "moondream"]
VISION_MODEL = None

for _candidate in VISION_CANDIDATES:
    try:
        chat(model=_candidate,
             messages=[{"role": "user", "content": "Reply with the word ok.",
                        "images": [str(DEMO_PATH)]}],
             options={"temperature": 0.0, "num_predict": 1})
        VISION_MODEL = _candidate
        print(f"vision model: {_candidate}")
        break
    except Exception as err:
        print(f"  {_candidate} unavailable ({type(err).__name__}: {str(err)[:80]})")

if VISION_MODEL is None:
    raise RuntimeError("No vision model loaded. Check that setup cell 3 finished.")
if VISION_MODEL != "llama3.2-vision":
    print(f"NOTE: falling back to {VISION_MODEL}. Say so in the report - the "
          f"brief names llama3.2-vision, so the substitution needs stating.")

In [ ]:
# --- Naive vs structured prompt, same image, same temperature ---
naive_out = ask_vision(PROMPT_T1_NAIVE, DEMO_PATH)
print("=== NAIVE PROMPT ===")
print(naive_out)

struct_out = ask_vision(PROMPT_T1_STRUCTURED, DEMO_PATH)
print("\n=== STRUCTURED PROMPT ===")
print(struct_out)

record_t1 = parse_model_json(struct_out)
print("\nparsed OK:", "error" not in record_t1)
pd.DataFrame([record_t1])

In [ ]:
# --- Stochasticity: three runs at temperature 0.8, then three at 0 ---
for temp in (0.8, 0.0):
    runs = [ask_vision(PROMPT_T1_STRUCTURED, DEMO_PATH, temperature=temp)
            for _ in range(3)]
    print(f"--- temperature={temp}: {len(set(runs))} distinct outputs of 3 ---")
    for i, r in enumerate(runs, 1):
        print(f"  run {i}: {r[:110].replace(chr(10), ' ')}...")
    print()

# Save everything the report needs to quote.
json.dump({"image_id": DEMO_ID, "naive_prompt": PROMPT_T1_NAIVE,
           "naive_response": naive_out,
           "structured_prompt": PROMPT_T1_STRUCTURED,
           "structured_response": struct_out, "parsed": record_t1},
          open("outputs/records/task1_vlm.json", "w"), indent=2)

## 4. Task 2a — classical features

Otsu threshold, morphological cleanup, connected components, then
`regionprops_table`. This is Lab 3's `analyse_cells` applied to our modality.
Everything here is deterministic: the same image always gives the same table,
which is what makes the downstream record auditable.

In [ ]:
# --- The classical pipeline (Lab 3, Exercise 7.1) ---
CLASSICAL_PROPS = ("label", "area", "perimeter", "eccentricity", "solidity",
                   "major_axis_length", "minor_axis_length", "mean_intensity")

def otsu_mask(gray_img, gauss_sigma=1.0, open_r=1, close_r=2, min_area=30):
    smooth = skfilters.gaussian(gray_img, sigma=gauss_sigma)
    T = skfilters.threshold_otsu(smooth)
    mask = smooth >= T
    mask = skmorphology.opening(mask, disk(open_r))
    mask = skmorphology.closing(mask, disk(close_r))
    mask = skmorphology.remove_small_objects(mask, min_size=min_area)
    return mask.astype(bool)

def region_table(gray_img, mask, min_area=30):
    """Binary mask -> per-object regionprops DataFrame."""
    mask = skmorphology.remove_small_objects(np.asarray(mask).astype(bool),
                                             min_size=min_area)
    labels = skmeasure.label(mask)
    if labels.max() == 0:
        return pd.DataFrame(columns=list(CLASSICAL_PROPS))
    df = pd.DataFrame(skmeasure.regionprops_table(
        labels, intensity_image=np.asarray(gray_img, dtype=float),
        properties=CLASSICAL_PROPS))
    # circularity = 4*pi*A/P^2 (1.0 for a circle). Left unclipped: skimage's
    # perimeter estimator under-measures small objects, pushing it above 1.
    per = df["perimeter"].replace(0, np.nan)
    df["circularity"] = 4 * np.pi * df["area"] / (per ** 2)
    return df.round(3)

def mask_dice(pred, gt, eps=1e-7):
    pred, gt = np.asarray(pred, bool), np.asarray(gt, bool)
    return float((2 * (pred & gt).sum() + eps) / (pred.sum() + gt.sum() + eps))

def mask_iou(pred, gt, eps=1e-7):
    pred, gt = np.asarray(pred, bool), np.asarray(gt, bool)
    inter = (pred & gt).sum()
    return float((inter + eps) / (pred.sum() + gt.sum() - inter + eps))

In [ ]:
# --- Figure 3: every stage, so you can see where objects are created or lost ---
img = load_gray(DATA / "train" / "images" / f"{DEMO_ID}.png")
gt = load_mask(DATA / "train" / "masks" / f"{DEMO_ID}.png").astype(bool)

smooth = skfilters.gaussian(img, sigma=1.0)
T = skfilters.threshold_otsu(smooth)
raw_mask = smooth >= T
cleaned = skmorphology.remove_small_objects(
    skmorphology.closing(skmorphology.opening(raw_mask, disk(1)), disk(2)), min_size=30)
labels = skmeasure.label(cleaned)

panels = [(img, f"grayscale input (mean {img.mean():.3f})", "gray"),
          (smooth, "Gaussian smoothed (sigma=1)", "gray"),
          (raw_mask, f"Otsu T={T:.3f}, {raw_mask.sum()} fg px", "gray"),
          (cleaned, f"cleaned, {cleaned.sum()} fg px", "gray"),
          (labels, f"{labels.max()} connected components", "nipy_spectral"),
          (gt, f"ground truth: {skmeasure.label(gt).max()} objects", "gray")]
fig, axes = plt.subplots(1, 6, figsize=(20, 3.6))
for ax, (im, title, cmap) in zip(axes, panels):
    ax.imshow(im, cmap=cmap); ax.set_title(title, fontsize=9); ax.axis("off")
fig.suptitle(f"Task 2 - classical pipeline stages on {DEMO_ID}")
fig.tight_layout()
fig.savefig("outputs/figures/fig3_otsu_stages.png", dpi=160, bbox_inches="tight")
plt.show()

feat = region_table(img, otsu_mask(img))
feat.to_csv("outputs/records/task2_features.csv", index=False)
print(f"feature table: {feat.shape[0]} objects x {feat.shape[1]} features")
feat.head()

In [ ]:
# --- How accurate is Otsu on its own? (needed for Question 2) ---
rows = []
for split in ["val", "test"]:
    for iid, ip, mp in list_split(split):
        im = load_gray(ip); g = load_mask(mp).astype(bool)
        pred = otsu_mask(im); df = region_table(im, pred)
        rows.append({"image_id": iid, "split": split,
                     "density": meta.set_index("image_id").loc[iid, "density"],
                     "gt_objects": int(meta.set_index("image_id").loc[iid, "n_objects"]),
                     "otsu_objects": len(df),
                     "otsu_dice": round(mask_dice(pred, g), 4),
                     "otsu_iou": round(mask_iou(pred, g), 4)})
otsu_df = pd.DataFrame(rows)
otsu_df["count_error"] = otsu_df.otsu_objects - otsu_df.gt_objects
otsu_df.to_csv("outputs/metrics/otsu_vs_truth.csv", index=False)

print(otsu_df.groupby("density")[["otsu_dice", "otsu_iou", "count_error"]].mean().round(3))
print(f"\noverall Dice={otsu_df.otsu_dice.mean():.4f}  IoU={otsu_df.otsu_iou.mean():.4f}")
print("\nNote the split verdict: pixel overlap is near-perfect, but the object")
print("count is badly wrong on dense and clustered fields. Otsu cannot separate")
print("touching nuclei - they merge into one connected component.")

## 5. Task 2b — the numbers-first description

The feature table is collapsed to one factual sentence, and *that sentence* is
all the language model receives. It never sees the image. Anything it states
that is not in the sentence is a hallucination by construction — which is
precisely what makes this route auditable and Task 1's route not.

In [ ]:
# --- Feature-to-text (Lab 5) and the Task 2 prompt ---
def summarise_features(df, image_name="image"):
    n = len(df)
    if n == 0:
        return f"In {image_name}, no objects were detected by the segmentation pipeline."
    a, e, s, i = df["area"], df["eccentricity"], df["solidity"], df["mean_intensity"]
    return (f"In {image_name}, the segmentation pipeline detected {n} objects. "
            f"Object areas range from {a.min():.0f} to {a.max():.0f} pixels "
            f"(mean {a.mean():.0f}, median {a.median():.0f}). "
            f"Mean eccentricity is {e.mean():.2f}; mean solidity is {s.mean():.2f}. "
            f"Mean object intensity is {i.mean():.2f}.")

PROMPT_T2_NUMBERS = '''You are writing the description section of a biomedical
image-analysis report. You have NOT seen the image. You have only the
measurements below, produced by a deterministic image-processing pipeline.

Based ONLY on those measurements:

1. Produce a JSON record with these fields:
   - n_objects (integer, copied exactly from the summary)
   - density_class ("sparse", "moderate", "dense")
   - shape_regularity ("highly irregular", "irregular", "regular", "highly regular")
   - quality_flag ("ok", "review_recommended", "fail")

2. Then a one-paragraph description (3-4 sentences) of the measured objects.

Format your response EXACTLY as:

JSON:
{{...}}

NARRATIVE:
<paragraph>

Rules:
- Every number you state must appear in the summary. Do NOT compute new
  numbers and do NOT invent any.
- Do NOT describe colour, staining, texture, or anything else you cannot know
  from measurements alone.
- Do NOT diagnose any medical condition.
- Interpret eccentricity as 0 = circular and 1 = elongated; solidity as
  1 = smooth convex boundary and lower = ragged or touching objects.

Measurements: {summary}'''

def _extract_json_block(text):
    """Return (json_string, remainder) by brace-matching the first {...} block."""
    start = text.find("{")
    if start == -1:
        return None, text
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1], text[i + 1:]
    return text[start:], ""


def parse_hybrid_response(text):
    """Split a combined JSON + narrative reply into (record, narrative).

    Deliberately tolerant. Splitting on a literal "NARRATIVE:" loses the
    paragraph whenever the model writes "**Narrative:**", "narrative -", or
    omits the marker and just starts the prose; in the first Colab run that cost
    9 of 12 narratives. Instead we brace-match the JSON object and treat
    everything after it as the narrative, stripping any marker we find.
    """
    text = text.strip()
    json_str, rest = _extract_json_block(text)
    if json_str is not None and rest.lstrip().startswith("```"):
        rest = rest.lstrip()[3:]          # closing fence of a ```json block
    record = parse_model_json(json_str) if json_str else {"error": "no JSON found"}
    rest = rest.strip().lstrip("`").strip()
    rest = re.sub(r"^[\s`*#_>-]*narrative[\s*:._>-]*", "", rest, flags=re.I)
    return record, rest.strip().strip("`").strip()

def ask_text(prompt, model="llama3.2", temperature=0.0):
    r = chat(model=model, messages=[{"role": "user", "content": prompt}],
             options={"temperature": temperature})
    return r["message"]["content"]

In [ ]:
# --- Run the numbers-first description and audit it ---
summary = summarise_features(feat, image_name=DEMO_ID)
print("SUMMARY SENT TO THE MODEL (this is all it sees):")
print(" ", summary, "\n")

raw_t2 = ask_text(PROMPT_T2_NUMBERS.format(summary=summary))
rec_t2, narr_t2 = parse_hybrid_response(raw_t2)
print("JSON record:", rec_t2)
print("\nNarrative:", narr_t2)

gt_n = int(meta.set_index("image_id").loc[DEMO_ID, "n_objects"])
print(f"\nAUDIT  measured={len(feat)}  LLM said={rec_t2.get('n_objects')}  "
      f"ground truth={gt_n}")
print("The LLM should match the MEASUREMENT exactly. The gap between the")
print("measurement and the ground truth is the segmentation's error, not the")
print("model's - keeping those two failures separable is the point.")

json.dump({"prompt": PROMPT_T2_NUMBERS.format(summary=summary),
           "summary": summary, "raw": raw_t2, "record": rec_t2,
           "narrative": narr_t2, "n_measured": len(feat), "n_truth": gt_n},
          open("outputs/records/task2_llm.json", "w"), indent=2)

## 6. Task 3 — the U-Net

The architecture, losses and metrics are exactly Lab 4's: a 3-level U-Net with
`DoubleConv` blocks and skip connections, roughly 483k parameters, trained with
Adam at `lr=1e-3`.

We train three of them — BCE, Dice, and BCE+Dice — with weight initialisation,
batch order and augmentation draws pinned to the same seed, so the only thing
that differs between runs is the loss. That covers the loss-ablation extension
at no extra cost, and gives the main model for Task 4.

**Runtime:** roughly 3 minutes per model on a T4, or 3 minutes on CPU at
128x128. Set `N_EPOCHS = 5` if you just want to see it work.

In [ ]:
# --- Dataset, architecture, losses, metrics (all from Lab 4) ---
class NucleiDataset(Dataset):
    def __init__(self, split, size=UNET_SIZE, augment=False):
        self.augment = augment
        self.images, self.masks, self.ids = [], [], []
        for iid, ip, mp in list_split(split):
            self.images.append(load_gray(ip, size=size, normalise=True))
            self.masks.append(load_mask(mp, size=size))
            self.ids.append(iid)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img, msk = self.images[idx].copy(), self.masks[idx].copy()
        if self.augment:
            if np.random.rand() < 0.5:
                img, msk = np.fliplr(img).copy(), np.fliplr(msk).copy()
            if np.random.rand() < 0.5:
                img, msk = np.flipud(img).copy(), np.flipud(msk).copy()
            k = np.random.randint(4)
            img, msk = np.rot90(img, k).copy(), np.rot90(msk, k).copy()
        return torch.from_numpy(img).unsqueeze(0), torch.from_numpy(msk).unsqueeze(0)


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()
        self.enc1, self.enc2 = DoubleConv(in_ch, base), DoubleConv(base, base*2)
        self.enc3 = DoubleConv(base*2, base*4)
        self.bottleneck = DoubleConv(base*4, base*8)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConv(base*2, base)
        self.out_conv, self.pool = nn.Conv2d(base, out_ch, 1), nn.MaxPool2d(2)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out_conv(d1)


def dice_loss(logits, target, eps=1e-7):
    probs = torch.sigmoid(logits)
    inter = (probs * target).sum(dim=(1, 2, 3))
    union = probs.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()

def bce_loss(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target)

def combined_loss(logits, target):
    return bce_loss(logits, target) + dice_loss(logits, target)

LOSSES = {"BCE": bce_loss, "Dice": dice_loss, "BCE+Dice": combined_loss}

def dice_coefficient(logits, target, threshold=0.5, eps=1e-7):
    preds = (torch.sigmoid(logits) > threshold).float()
    inter = (preds * target).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((2 * inter + eps) / (union + eps)).mean().item()

def iou_score(logits, target, threshold=0.5, eps=1e-7):
    preds = (torch.sigmoid(logits) > threshold).float()
    inter = (preds * target).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - inter
    return ((inter + eps) / (union + eps)).mean().item()

print("U-Net parameters:", sum(p.numel() for p in UNet().parameters()))

In [ ]:
# --- Train all three losses under identical conditions ---
N_EPOCHS, BATCH_SIZE, LR = 20, 8, 1e-3

train_ds = NucleiDataset("train", augment=True)
val_ds = NucleiDataset("val", augment=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

def train_model(loss_fn, label, seed=0, n_epochs=N_EPOCHS):
    np.random.seed(seed); torch.manual_seed(seed)   # identical init + aug draws
    model = UNet(base=16).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    g = torch.Generator(); g.manual_seed(seed)      # identical batch order
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g)

    hist = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": []}
    for epoch in range(n_epochs):
        model.train(); running = 0.0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = loss_fn(model(x), y)
            loss.backward(); opt.step()
            running += loss.item() * x.size(0)
        running /= len(train_ds)

        model.eval(); v_loss = v_dice = v_iou = n = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x); bs = x.size(0)
                v_loss += loss_fn(logits, y).item() * bs
                v_dice += dice_coefficient(logits, y) * bs
                v_iou += iou_score(logits, y) * bs
                n += bs
        hist["train_loss"].append(running); hist["val_loss"].append(v_loss/n)
        hist["val_dice"].append(v_dice/n); hist["val_iou"].append(v_iou/n)
        print(f"  [{label:<9}] epoch {epoch+1:2d}/{n_epochs}  train={running:.4f}  "
              f"val_Dice={v_dice/n:.4f}", flush=True)
    return model, hist

models, histories, ablation_rows = {}, {}, []
for name, fn in LOSSES.items():
    m, h = train_model(fn, name)
    models[name], histories[name] = m, h
    ablation_rows.append({"loss": name, "val_dice": round(h["val_dice"][-1], 4),
                          "val_iou": round(h["val_iou"][-1], 4),
                          "final_train_loss": round(h["train_loss"][-1], 4)})
    torch.save(m.state_dict(), f"models/unet_{name.replace('+','_')}.pth")
    print()

ablation = pd.DataFrame(ablation_rows)
ablation.to_csv("outputs/metrics/unet_loss_ablation.csv", index=False)
json.dump(histories, open("outputs/metrics/unet_history.json", "w"), indent=2)

model = models["BCE+Dice"]     # main model for Task 4
torch.save(model.state_dict(), "models/unet_main.pth")
ablation

In [ ]:
# --- Figure 4: loss curves, Dice curves, ablation ---
colours = {"BCE": "#4C72B0", "Dice": "#DD8452", "BCE+Dice": "#55A868"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

for name, h in histories.items():
    e = range(1, len(h["train_loss"]) + 1)
    axes[0].plot(e, h["train_loss"], color=colours[name], label=f"{name} (train)")
    axes[0].plot(e, h["val_loss"], "--", color=colours[name], alpha=0.7,
                 label=f"{name} (val)")
    axes[1].plot(e, h["val_dice"], color=colours[name], label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (own scale per model)")
axes[0].set_title("(a) Loss curves"); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation Dice")
axes[1].set_title("(b) Validation Dice - the only comparable curve")
axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

x = np.arange(len(ablation))
axes[2].bar(x - 0.2, ablation.val_dice, 0.4, label="Dice", color="#55A868")
axes[2].bar(x + 0.2, ablation.val_iou, 0.4, label="IoU", color="#4C72B0")
axes[2].set_xticks(x); axes[2].set_xticklabels(ablation.loss); axes[2].set_ylim(0.8, 1.0)
axes[2].set_title("(c) Loss ablation (final epoch)"); axes[2].legend(fontsize=8)
fig.suptitle(f"Task 3 - U-Net training and loss ablation ({N_EPOCHS} epochs, {UNET_SIZE}px)")
fig.tight_layout()
fig.savefig("outputs/figures/fig4_curves.png", dpi=160, bbox_inches="tight")
plt.show()

print("The loss curves live on different scales and must not be compared to")
print("each other - only the Dice curve is comparable across the three runs.")

In [ ]:
# --- segment_image: the Lab 4 inference contract, reused by Task 4 ---
@torch.no_grad()
def segment_image(model, image, size=UNET_SIZE, threshold=0.5, out_size=None,
                  return_prob=False):
    """Always returns a uint8 {0,1} mask of shape (out_size, out_size).

    out_size lets Task 4 get the mask back at 256 so that regionprops areas are
    in the same pixel units as the ground-truth metadata.
    """
    out_size = size if out_size is None else out_size
    empty = np.zeros((out_size, out_size), np.uint8)
    device = next(model.parameters()).device

    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy()
    arr = np.asarray(image)
    if arr.ndim == 0 or arr.size == 0:
        return (empty, np.zeros((out_size, out_size), np.float32)) if return_prob else empty
    if arr.ndim == 1:
        arr = arr[None, :]
    elif arr.ndim == 3:
        arr = rgb2gray(arr[..., :3]) if arr.shape[-1] in (3, 4) else (
            arr[:3].mean(axis=0) if arr.shape[0] in (1, 3, 4) else arr.max(axis=0))
    elif arr.ndim > 3:
        arr = arr.reshape(-1, *arr.shape[-2:]).max(axis=0)

    arr = np.nan_to_num(np.asarray(arr, np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if arr.shape != (size, size):
        arr = resize(arr, (size, size), anti_aliasing=True,
                     preserve_range=True).astype(np.float32)
    arr = to_unit_range(arr)

    was_training = model.training
    model.eval()
    prob = torch.sigmoid(model(torch.from_numpy(arr)[None, None].to(device)))
    model.train(was_training)
    prob = prob[0, 0].cpu().numpy()
    if out_size != size:
        prob = resize(prob, (out_size, out_size), order=0,
                      anti_aliasing=False, preserve_range=True)
    mask = (prob > threshold).astype(np.uint8)
    return (mask, prob) if return_prob else mask

# Sanity check: awkward inputs must not crash.
for name, im in [("val sample", val_ds[0][0]), ("all black", np.zeros((128, 128))),
                 ("uint8 RGB", np.random.randint(0, 256, (200, 180, 3), np.uint8)),
                 ("empty", np.array([]))]:
    m = segment_image(model, im)
    print(f"{name:<12} -> {m.shape} {m.dtype}  {100*m.mean():.1f}% foreground")

In [ ]:
# --- Per-image evaluation: U-Net vs Otsu, pixel overlap AND object count ---
mi = meta.set_index("image_id")
rows = []
for iid, ip, mp in list_split("val"):
    img256 = load_gray(ip)
    gt256 = load_mask(mp).astype(bool)
    gt128 = load_mask(mp, size=UNET_SIZE).astype(bool)
    pred128 = segment_image(model, load_gray(ip, size=UNET_SIZE, normalise=True)).astype(bool)
    pred256 = segment_image(model, img256, out_size=256).astype(bool)
    otsu = otsu_mask(img256)
    rows.append({"image_id": iid, "density": mi.loc[iid, "density"],
                 "gt_objects": int(mi.loc[iid, "n_objects"]),
                 "unet_dice": round(mask_dice(pred128, gt128), 4),
                 "unet_iou": round(mask_iou(pred128, gt128), 4),
                 "unet_dice_256": round(mask_dice(pred256, gt256), 4),
                 "unet_iou_256": round(mask_iou(pred256, gt256), 4),
                 "otsu_dice": round(mask_dice(otsu, gt256), 4),
                 "otsu_iou": round(mask_iou(otsu, gt256), 4),
                 "unet_objects": len(region_table(img256, pred256)),
                 "otsu_objects": len(region_table(img256, otsu))})
per_image = pd.DataFrame(rows)
per_image["unet_minus_otsu"] = (per_image.unet_dice_256 - per_image.otsu_dice).round(4)
per_image.to_csv("outputs/metrics/unet_per_image.csv", index=False)

print(per_image.groupby("density")[["unet_dice_256", "otsu_dice", "unet_objects",
                                    "otsu_objects", "gt_objects"]].mean().round(3))
print(f"\nU-Net@128 Dice={per_image.unet_dice.mean():.4f} IoU={per_image.unet_iou.mean():.4f}")
print(f"U-Net@256 Dice={per_image.unet_dice_256.mean():.4f} IoU={per_image.unet_iou_256.mean():.4f}")
print(f"Otsu@256  Dice={per_image.otsu_dice.mean():.4f} IoU={per_image.otsu_iou.mean():.4f}")
print("\nThe U-Net is scored twice on purpose. Comparing a 128px prediction")
print("against a 256px Otsu mask would confound the model with the resolution.")

In [ ]:
# --- Figure 5: worst two and best two validation images, with an error map ---
order = per_image.sort_values("unet_dice")
picks = list(order.head(2).image_id) + list(order.tail(2).image_id)

fig, axes = plt.subplots(len(picks), 4, figsize=(13, 3.2 * len(picks)))
for r, iid in enumerate(picks):
    im = load_gray(DATA / "val" / "images" / f"{iid}.png", size=UNET_SIZE, normalise=True)
    g = load_mask(DATA / "val" / "masks" / f"{iid}.png", size=UNET_SIZE).astype(bool)
    p = segment_image(model, im).astype(bool)
    err = np.zeros((*g.shape, 3))
    err[..., 0] = p & ~g      # false positive - red
    err[..., 1] = g & p       # correct - green
    err[..., 2] = g & ~p      # false negative - blue
    row = per_image[per_image.image_id == iid].iloc[0]
    for c, (arr, title, cmap) in enumerate([
            (im, f"{iid} ({row.density})", "gray"),
            (g, f"ground truth - {row.gt_objects} nuclei", "gray"),
            (p, f"U-Net - Dice {row.unet_dice:.3f}", "gray"),
            (err, "green=hit  red=FP  blue=FN", None)]):
        axes[r, c].imshow(arr, cmap=cmap); axes[r, c].set_title(title, fontsize=9)
        axes[r, c].axis("off")
fig.suptitle("Task 3 - worst two and best two validation images")
fig.tight_layout()
fig.savefig("outputs/figures/fig5_predictions.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 6: the two metrics disagree about which method is better ---
pi = per_image.copy()
pi["ue"] = (pi.unet_objects - pi.gt_objects).abs()
pi["oe"] = (pi.otsu_objects - pi.gt_objects).abs()
pi["count_gain"] = pi.oe - pi.ue

cases = [(pi.sort_values("count_gain").iloc[-1], "U-Net closer on object count"),
         (pi.sort_values("unet_minus_otsu").iloc[0], "Otsu clearly higher Dice")]

fig, axes = plt.subplots(2, 4, figsize=(13, 6.6))
for r, (row, winner) in enumerate(cases):
    iid = row.image_id
    im = load_gray(DATA / "val" / "images" / f"{iid}.png")
    g = load_mask(DATA / "val" / "masks" / f"{iid}.png").astype(bool)
    u = segment_image(model, im, out_size=256).astype(bool)
    o = otsu_mask(im)
    for c, (arr, title) in enumerate([
            (im, f"{iid} ({row.density}) - {winner}"),
            (g, f"ground truth - {row.gt_objects} nuclei"),
            (u, f"U-Net: Dice {row.unet_dice_256:.3f}, {row.unet_objects} obj"),
            (o, f"Otsu: Dice {row.otsu_dice:.3f}, {row.otsu_objects} obj")]):
        axes[r, c].imshow(arr, cmap="gray"); axes[r, c].set_title(title, fontsize=9)
        axes[r, c].axis("off")
fig.suptitle("Task 3 - pixel overlap and object count disagree about the winner")
fig.tight_layout()
fig.savefig("outputs/figures/fig6_unet_vs_otsu.png", dpi=160, bbox_inches="tight")
plt.show()

print(f"U-Net counts closer on {(pi.ue < pi.oe).sum()}/20, equal on {(pi.ue == pi.oe).sum()}")
print(f"Otsu has the higher Dice on {(pi.otsu_dice > pi.unet_dice_256).sum()}/20")

## 7. Task 4 — the hybrid pipeline

```
test image -> U-Net mask -> regionprops -> summary -> LLM (JSON + narrative) -> CSV
```

Two additions from Lab 5 that turn this from a demo into something auditable:

* a **quality gate** on the measurements, *before* the LLM is called. Cheap
  deterministic checks reject a bad mask without spending an LLM call, so an
  image full of speckle never reaches the stage that would narrate it.
* an **audit column**, `n_objects_measured`, copied straight from
  `regionprops`. If it ever disagrees with the LLM's `n_objects`, the model has
  hallucinated and the row flags itself.

In [ ]:
# --- Quality gate and the full pipeline ---
PROMPT_T4_HYBRID = '''You are summarising a biomedical image for a research report.

You will be given a textual summary of measurements taken from a segmentation
mask. Based ONLY on that summary:

1. Produce a JSON record with these fields:
   - image_id (string, copied exactly from the summary)
   - n_objects (integer, copied exactly from the summary)
   - mean_area (number, copied exactly from the summary)
   - density_class ("sparse", "moderate", "dense")
   - quality_flag ("ok", "review_recommended", "fail")

2. Then a one-paragraph narrative (3-4 sentences) suitable for a research report.

Format your response EXACTLY as:

JSON:
{{...}}

NARRATIVE:
<paragraph>

Rules:
- Do NOT invent details that are not in the summary.
- Do NOT diagnose any medical condition.
- Copy n_objects and mean_area verbatim; they are measurements, not estimates.
- If the summary reports no objects, or object counts or sizes that look
  implausible for cell nuclei, set quality_flag to "fail".

Image ID: {image_id}
Summary: {summary}'''

def quality_gate(df):
    """Pass/fail from the measurement table alone. Deterministic and cheap.

    Thresholds come from the training distribution: 5-85 nuclei per image with
    areas of roughly 80-400 px at 256x256.
    """
    n = len(df)
    if n == 0:
        return False, "no objects detected"
    if n > 150:
        return False, "too many objects (likely noise / over-segmentation)"
    if df["area"].mean() < 20:
        return False, "mean object area too small (mask is mostly speckle)"
    if df["area"].mean() > 5000:
        return False, "mean object area implausibly large (mask likely merged)"
    return True, "ok"

def full_pipeline(model, image, image_id, model_name="llama3.2", gate=True):
    mask = segment_image(model, image, out_size=256)          # stage 1
    df = region_table(image, mask)                            # stage 2
    if gate:
        passed, reason = quality_gate(df)
        if not passed:
            return {"image_id": image_id, "n_objects": len(df),
                    "mean_area": round(float(df["area"].mean()), 1) if len(df) else 0.0,
                    "density_class": "unknown", "quality_flag": "fail",
                    "gate_reason": reason, "llm_called": False,
                    "n_objects_measured": len(df),
                    "narrative": "Image rejected at quality gate; not narrated."}, df, mask
    summary = summarise_features(df, image_name=image_id)      # stage 3
    raw = ask_text(PROMPT_T4_HYBRID.format(image_id=image_id, summary=summary),
                   model=model_name)                           # stage 4
    record, narrative = parse_hybrid_response(raw)             # stage 5
    record.update({"image_id": image_id, "gate_reason": "ok", "llm_called": True,
                   "n_objects_measured": len(df),
                   "mean_area_measured": round(float(df["area"].mean()), 1) if len(df) else 0.0,
                   "narrative": narrative})
    return record, df, mask

In [ ]:
# --- Run on the 12 unseen test images and aggregate ---
records = []
for iid, ip, mp in list_split("test"):
    im = load_gray(ip); g = load_mask(mp).astype(bool)
    rec, df, mask = full_pipeline(model, im, iid)
    rec["gt_objects"] = int(mi.loc[iid, "n_objects"])
    rec["density_truth"] = mi.loc[iid, "density"]
    rec["unet_dice_vs_gt"] = round(mask_dice(mask.astype(bool), g), 4)
    rec["count_hallucinated"] = rec.get("n_objects") != rec.get("n_objects_measured")
    records.append(rec)
    print(f"{iid}: measured {rec['n_objects_measured']}, "
          f"LLM said {rec.get('n_objects')}, truth {rec['gt_objects']}")

task4 = pd.DataFrame(records)
task4.to_csv("outputs/records/task4_records.csv", index=False)
print(f"\nHallucinated counts: {int(task4.count_hallucinated.sum())}/{len(task4)}")
print(f"Mean U-Net Dice on the test split: {task4.unet_dice_vs_gt.mean():.4f}")
task4[["image_id", "n_objects_measured", "n_objects", "gt_objects",
       "density_class", "quality_flag", "unet_dice_vs_gt", "count_hallucinated"]]

In [ ]:
# --- An example record and its narrative, for the report ---
ex = records[4]
print("JSON record:")
print(json.dumps({k: v for k, v in ex.items() if not k.startswith("_")
                  and k != "narrative"}, indent=2))
print("\nNarrative:")
print(ex["narrative"])

# Markdown report (the human-readable deliverable alongside the CSV)
n_flag = int((task4.quality_flag != "ok").sum())
parts = ["# Hybrid pipeline report - unseen test split\n\n",
         f"**Images:** {len(task4)} | **Passed:** {len(task4)-n_flag} | "
         f"**Flagged:** {n_flag}\n\n---\n\n"]
for r in records:
    status = "OK" if r.get("quality_flag") == "ok" else "FLAGGED"
    parts.append(f"## {r['image_id']} - {status}\n\n")
    parts.append(f"- measured objects: {r['n_objects_measured']} "
                 f"(ground truth {r['gt_objects']}, regime {r['density_truth']})\n")
    parts.append(f"- U-Net Dice: {r['unet_dice_vs_gt']}\n")
    parts.append(f"- count matches measurement: {not r['count_hallucinated']}\n\n")
    parts.append(f"{r['narrative']}\n\n")
open("outputs/records/task4_report.md", "w").write("".join(parts))
print("\nSaved outputs/records/task4_report.md")

## 8. Extension — robustness

The dataset ships blurred and low-contrast variants of `test_000` and
`test_004`. We push all six images (2 clean + 4 corrupted) through every stage
and record what each stage sees, so we can identify the *earliest* point at
which the corruption is detectable.

The two corruptions are not equivalent. Blur (sigma=6) destroys boundaries and
merges neighbours — an irreversible loss of spatial information. Low contrast
is a global affine rescale, `(I - 0.5) * 0.15 + 0.5`, which the per-image
min-max normalisation inside `segment_image` simply undoes. Predict which one
the pipeline survives before you run the cell.

In [ ]:
# --- Trace both corruptions through every stage ---
corr = DATA / "test_corrupted" / "images"
cases = []
for base in ["test_000", "test_004"]:
    cases += [(base, "clean", DATA / "test" / "images" / f"{base}.png"),
              (base, "blur", corr / f"{base}_blur.png"),
              (base, "lowcontrast", corr / f"{base}_lowcontrast.png")]

rows, panels = [], []
for base, kind, path in cases:
    im = load_gray(path)
    g = load_mask(DATA / "test" / "masks" / f"{base}.png").astype(bool)
    mask = segment_image(model, im, out_size=256)
    df = region_table(im, mask)
    passed, reason = quality_gate(df)
    rec, _, _ = full_pipeline(model, im, f"{base}_{kind}")
    rows.append({"image_id": base, "corruption": kind,
                 "stage0_img_std": round(float(im.std()), 4),
                 "stage1_fg_fraction": round(float(mask.mean()), 4),
                 "stage1_dice_vs_gt": round(mask_dice(mask.astype(bool), g), 4),
                 "stage2_n_objects": len(df),
                 "stage2_mean_area": round(float(df["area"].mean()), 1) if len(df) else 0.0,
                 "gt_objects": int(mi.loc[base, "n_objects"]),
                 "gate_passed": passed, "gate_reason": reason,
                 "stage4_quality_flag": rec.get("quality_flag")})
    panels.append((f"{base}\n{kind}", im, mask, len(df)))

trace = pd.DataFrame(rows)
trace.to_csv("outputs/metrics/robustness_trace.csv", index=False)

fig, axes = plt.subplots(2, 6, figsize=(19, 6.4))
for c, (title, im, mask, n) in enumerate(panels):
    r, cc = divmod(c, 3)
    axes[r, cc*2].imshow(im, cmap="gray", vmin=0, vmax=float(max(im.max(), 1e-6)))
    axes[r, cc*2].set_title(title, fontsize=9)
    axes[r, cc*2+1].imshow(mask, cmap="gray")
    axes[r, cc*2+1].set_title(f"U-Net mask - {n} objects", fontsize=9)
    axes[r, cc*2].axis("off"); axes[r, cc*2+1].axis("off")
fig.suptitle("Extension - how blur and low contrast propagate through the pipeline")
fig.tight_layout()
fig.savefig("outputs/figures/fig7_robustness.png", dpi=160, bbox_inches="tight")
plt.show()
trace

## 9. Extension — vision-model comparison

Same image, same structured prompt, three vision models. What we care about is
not which prose reads best but which model **obeys the schema**: does it return
parseable JSON, does it stay inside the closed vocabularies, and does it use
`"uncertain"` when it should rather than inventing a confident answer.

In [ ]:
# --- Compare vision models on the Task 1 description step ---
VISION_MODELS = [m for m in ["llama3.2-vision", "qwen2.5vl:7b", "moondream"]
                 if m == VISION_MODEL or m != "llama3.2-vision"]
rows = []
for m_name in VISION_MODELS:
    try:
        out = ask_vision(PROMPT_T1_STRUCTURED, DEMO_PATH, model=m_name)
        rec = parse_model_json(out)
        rows.append({"model": m_name,
                     "json_parsed": "error" not in rec,
                     "modality": rec.get("modality"),
                     "tissue_type": rec.get("tissue_type"),
                     "image_quality": rec.get("image_quality"),
                     "n_features": len(rec.get("notable_features", []) or []),
                     "chars": len(out)})
    except Exception as err:
        rows.append({"model": m_name, "json_parsed": False,
                     "modality": f"ERROR: {type(err).__name__}"})
comparison = pd.DataFrame(rows)
comparison.to_csv("outputs/metrics/vision_model_comparison.csv", index=False)
comparison

## 10. Wrap-up

Files produced, all under `outputs/`:

* `figures/fig1`–`fig7` — every figure the report uses
* `metrics/` — EDA summary, Otsu accuracy, loss ablation, per-image scores,
  robustness trace, vision-model comparison
* `records/task4_records.csv` — **the deliverable**: one structured JSON record
  per unseen test image, with the measured object count alongside the LLM's, so
  any hallucination is visible in the table itself
* `records/task4_report.md` — the human-readable version

The single most important column in that CSV is `n_objects_measured`. It comes
from `regionprops`, not from a language model, and it is what makes every claim
in the narrative checkable.

In [ ]:
# --- Optional: download the deliverables ---
!zip -qr assignment3_outputs.zip outputs models
from google.colab import files
files.download("assignment3_outputs.zip")